recommandation system

In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = '/Users/melina/Downloads/finalpaperdataset.csv'
df = pd.read_csv(file_path)

X = df.drop(columns='Decision')
y = df['Decision']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.30,      
    random_state=42, 
    stratify=y           
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50,      
    random_state=42, 
    stratify=y_temp
)

print(f"Train set shape: {X_train.shape}, Validation set shape: {X_val.shape}, Test set shape: {X_test.shape}")
print(f"Train target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Validation target distribution:\n{y_val.value_counts(normalize=True)}")
print(f"Test target distribution:\n{y_test.value_counts(normalize=True)}")


Train set shape: (9100, 15), Validation set shape: (1950, 15), Test set shape: (1950, 15)
Train target distribution:
Decision
1    0.536264
0    0.463736
Name: proportion, dtype: float64
Validation target distribution:
Decision
1    0.53641
0    0.46359
Name: proportion, dtype: float64
Test target distribution:
Decision
1    0.53641
0    0.46359
Name: proportion, dtype: float64


In [38]:
df['Applicant_ID'] = range(1, len(df) + 1)
df.head(2)

,Decision,Applicant_GPA,Decision_Year,Target_University_FSR_Score,Target_University_CPF_Score,Target_University_ISR_Score,Target_University_QS_Rank,Target_Degree_Type,Applicant_Status,Target_University_Status,Target_Program,Target_Program_Discipline,Target_University,Target_Country,Program_Admission_Rate,Program_x_QS_Rank,Applicant_ID
0,1,-3.302371,1.316414,-0.128730,1.363850,-1.107856,-0.671909,1,1.0,1.0,0.027231,0.070154,0.005154,0.001077,0.629944,-0.018297,1
1,1,-1.853384,1.316414,-1.148255,0.595068,1.191774,-0.623350,1,1.0,1.0,0.010385,0.010385,0.003385,0.001077,0.570370,-0.006473,2


In [39]:


import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, f1_score

xgb_base = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=8,
    colsample_bytree=0.9,     
    subsample=0.9,
    reg_alpha=0.1,
    reg_lambda=1.5,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_calibrated = CalibratedClassifierCV(
    base_estimator=xgb_base,
    method='sigmoid',   
    cv=5
)

xgb_calibrated.fit(X_train, y_train)

y_pred_xgb_test = xgb_calibrated.predict(X_test)
y_prob_xgb_test = xgb_calibrated.predict_proba(X_test)[:, 1]

threshold = 0.41

misclassified_idx = np.where(y_pred_xgb_test != y_test)[0]
borderline_idx = np.where(
    (y_prob_xgb_test > threshold) &
    (y_prob_xgb_test < 1 - threshold)
)[0]

residual_idx_test = np.unique(np.concatenate([misclassified_idx, borderline_idx]))

X_residual_test = X_test.iloc[residual_idx_test].copy()
X_residual_test["xgb_prob"] = y_prob_xgb_test[residual_idx_test]


y_pred_xgb_train = xgb_calibrated.predict(X_train)
y_prob_xgb_train = xgb_calibrated.predict_proba(X_train)[:, 1]

misclassified_idx_train = np.where(y_pred_xgb_train != y_train)[0]
borderline_idx_train = np.where(
    (y_prob_xgb_train > threshold) &
    (y_prob_xgb_train < 1 - threshold)
)[0]

residual_idx_train = np.unique(np.concatenate([misclassified_idx_train, borderline_idx_train]))

X_residual_train = X_train.iloc[residual_idx_train].copy()
X_residual_train["xgb_prob"] = y_prob_xgb_train[residual_idx_train]
y_residual_train = y_train.iloc[residual_idx_train]

knn_best = KNeighborsClassifier(
    n_neighbors=11,
    weights="uniform"
)

knn_best.fit(X_residual_train, y_residual_train)

y_hybrid_test = y_pred_xgb_test.copy()
y_hybrid_test[residual_idx_test] = knn_best.predict(X_residual_test)

print("XGBoost (Platt-calibrated) Accuracy (test):",
      accuracy_score(y_test, y_pred_xgb_test))

print("Hybrid XGBoost + kNN Accuracy (test):",
      accuracy_score(y_test, y_hybrid_test))

print("Hybrid F1-score:",
      f1_score(y_test, y_hybrid_test))

print("\nClassification Report – Hybrid Model:\n")
print(classification_report(
    y_test,
    y_hybrid_test,
    target_names=["Rejected", "Accepted"]
))


/opt/anaconda3/lib/python3.11/site-packages/sklearn/calibration.py:321: FutureWarning: `base_estimator` was renamed to `estimator` in version 1.2 and will be removed in 1.4.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [05:54:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [05:54:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [05:54:29] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboos

XGBoost (Platt-calibrated) Accuracy (test): 0.717948717948718
Hybrid XGBoost + kNN Accuracy (test): 0.8646153846153846
Hybrid F1-score: 0.8734419942473634

Classification Report – Hybrid Model:

              precision    recall  f1-score   support

    Rejected       0.85      0.86      0.85       904
    Accepted       0.88      0.87      0.87      1046

    accuracy                           0.86      1950
   macro avg       0.86      0.86      0.86      1950
weighted avg       0.86      0.86      0.86      1950



In [40]:


import numpy as np
import pandas as pd

X_df = df[X_train.columns] 
xgb_probs_df = xgb_calibrated.predict_proba(X_df)[:, 1]

threshold = 0.41
hybrid_probs_df = xgb_probs_df.copy()

borderline_mask = (xgb_probs_df > threshold) & (xgb_probs_df < 1 - threshold)
if borderline_mask.any():
    knn_inputs_df = X_df[borderline_mask].copy()
    knn_inputs_df["xgb_prob"] = xgb_probs_df[borderline_mask]
    hybrid_probs_df[borderline_mask] = knn_best.predict(knn_inputs_df)

df["probability_of_acceptance"] = hybrid_probs_df

df.head()


,Decision,Applicant_GPA,Decision_Year,Target_University_FSR_Score,Target_University_CPF_Score,Target_University_ISR_Score,Target_University_QS_Rank,Target_Degree_Type,Applicant_Status,Target_University_Status,Target_Program,Target_Program_Discipline,Target_University,Target_Country,Program_Admission_Rate,Program_x_QS_Rank,Applicant_ID,probability_of_acceptance
0,1,-3.302371,1.316414,-0.128730,1.363850,-1.107856,-0.671909,1,1.0,1.0,0.027231,0.070154,0.005154,0.001077,0.629944,-0.018297,1,0.837152
1,1,-1.853384,1.316414,-1.148255,0.595068,1.191774,-0.623350,1,1.0,1.0,0.010385,0.010385,0.003385,0.001077,0.570370,-0.006473,2,0.686783
2,1,0.429870,1.316414,-1.381537,-0.052327,-0.159258,0.050401,1,2.0,1.0,0.083308,0.087077,0.010000,0.845308,0.541090,0.004199,3,0.854386
3,1,-0.228761,1.316414,1.224878,0.944515,1.115120,-0.802410,1,2.0,1.0,0.082231,0.139077,0.005308,0.035846,0.670720,-0.065983,4,0.758738
4,0,0.825048,1.316414,1.147118,0.558285,1.105538,-0.781166,1,2.0,2.0,0.006308,0.006308,0.022154,0.845308,0.597561,-0.004927,5,0.000000


In [41]:
import pickle

with open("freq_mappingsF.pkl", "rb") as f:
    freq_mappings = pickle.load(f)

print("Type of freq_mappings:", type(freq_mappings))

if isinstance(freq_mappings, dict):
    first_key = next(iter(freq_mappings))
    first_value = freq_mappings[first_key]
    print("Sample entry:")
    print("Key =", first_key)
    print("Value =", first_value)
    print("Value type =", type(first_value))
else:
    print("Sample entry:", freq_mappings[0])
    print("Type:", type(freq_mappings[0]))


Type of freq_mappings: <class 'dict'>
Sample entry:
Key = Target_Program
Value = {'Computer Science': 0.0833076923076923, 'Food and Resource Economics': 0.08223076923076923, 'Economics': 0.05515384615384615, 'Physics': 0.04138461538461539, 'Speech-Language Pathology': 0.039846153846153844, 'Chemistry': 0.03346153846153846, 'Philosophy': 0.03053846153846154, 'Public Policy & Management': 0.027230769230769232, 'English': 0.02676923076923077, 'Political Science': 0.023076923076923078, 'Urban & Regional Planning': 0.022, 'Mechanical Engineering': 0.02023076923076923, 'Mathematics': 0.01953846153846154, 'Systems Engineering': 0.01946153846153846, 'Electrical & Computer Engineering': 0.01876923076923077, 'Sociology': 0.01853846153846154, 'History': 0.017153846153846155, 'Statistics': 0.016307692307692308, 'Clinical Psychology': 0.015923076923076922, 'Education Policy & Leadership': 0.012538461538461538, 'Astronomy': 0.011692307692307693, 'Chemical Engineering': 0.010846153846153846, 'Geograp

In [42]:
print(freq_mappings.keys())


dict_keys(['Target_Program', 'Target_Program_Discipline', 'Target_University', 'Target_Country'])


In [43]:

with open("scalerF.pkl", "rb") as f:
    scaler = pickle.load(f)
print("Scaler Mean:", scaler.mean_)  
print("Scaler Scale:", scaler.scale_)  


Scaler Mean: [   3.75209923   57.46975385   62.92255385   58.18628462  270.39280769
 2022.97238462]
Scaler Scale: [2.27745204e-01 3.47220246e+01 2.71858629e+01 3.13093788e+01
 3.29498327e+02 1.54025687e+00]


### Revised Matching Strategy for the University Recommendation Module

Following an empirical analysis of recommendation sparsity, the initial exact-matching strategy was found to be overly restrictive, resulting in a substantial proportion of applicants receiving no alternative university recommendations. To address this limitation while preserving methodological rigor, the matching criteria were revised as follows.

First, **Target Program** is retained as a fixed attribute; however, instead of enforcing exact string or encoded equality, a **regex-based (approximate) matching strategy** is adopted. This allows semantically equivalent programs (e.g., variations in naming conventions or abbreviations) to be matched, thereby improving recall without compromising program-level relevance.

Second, **Applicant GPA** is no longer matched exactly. Given that GPA is a continuous variable and subject to scaling transformations, exact equality is both unrealistic and statistically inefficient. Instead, applicants are matched within a predefined **GPA tolerance range**, enabling the identification of historically comparable profiles while accounting for minor numerical variations.

Third, **Target Degree Type** (e.g., Bachelor’s, Master’s, PhD) remains fixed. Degree type fundamentally constrains admission requirements and institutional selectivity; relaxing this dimension would introduce conceptually invalid comparisons.

Importantly, **Applicant Status** and **Decision Year** are deliberately *not* fixed in the matching stage. These variables are already incorporated as predictive features within the hybrid XGBoost–kNN model and exert a significant influence on the estimated admission probability. Conditioning on them again during candidate selection would introduce unnecessary strictness, reduce candidate availability, and increase the proportion of missing recommendations without providing additional explanatory benefit.



Based on your dataset of high-GPA applicants, a narrow GPA range of ±0.1 around the reported GPA is appropriate for both Master’s and PhD applicants. PhD programs rely more on research and less on GPA, while Master’s programs still consider GPA moderately, so using the same threshold simplifies matching without losing accuracy.

In [44]:
import pandas as pd
import pickle

with open("scalerF.pkl", "rb") as f:
    scaler = pickle.load(f)

numeric_features = [
    'Applicant_GPA',
    'Target_University_FSR_Score',
    'Target_University_CPF_Score',
    'Target_University_ISR_Score',
    'Target_University_QS_Rank',
    'Decision_Year'
]

numeric_features = [col for col in numeric_features if col in df.columns]


df[numeric_features] = scaler.inverse_transform(df[numeric_features])


df.head()


,Decision,Applicant_GPA,Decision_Year,Target_University_FSR_Score,Target_University_CPF_Score,Target_University_ISR_Score,Target_University_QS_Rank,Target_Degree_Type,Applicant_Status,Target_University_Status,Target_Program,Target_Program_Discipline,Target_University,Target_Country,Program_Admission_Rate,Program_x_QS_Rank,Applicant_ID,probability_of_acceptance
0,1,3.00,2025.0,53.0,100.0,23.5,49.0,1,1.0,1.0,0.027231,0.070154,0.005154,0.001077,0.629944,-0.018297,1,0.837152
1,1,3.33,2025.0,17.6,79.1,95.5,65.0,1,1.0,1.0,0.010385,0.010385,0.003385,0.001077,0.570370,-0.006473,2,0.686783
2,1,3.85,2025.0,9.5,61.5,53.2,287.0,1,2.0,1.0,0.083308,0.087077,0.010000,0.845308,0.541090,0.004199,3,0.854386
3,1,3.70,2025.0,100.0,88.6,93.1,6.0,1,2.0,1.0,0.082231,0.139077,0.005308,0.035846,0.670720,-0.065983,4,0.758738
4,0,3.94,2025.0,97.3,78.1,92.8,13.0,1,2.0,2.0,0.006308,0.006308,0.022154,0.845308,0.597561,-0.004927,5,0.000000


In [45]:
import pandas as pd
import numpy as np
import pickle

with open('/Users/melina/Downloads/freq_mappingsF.pkl', 'rb') as f:
    freq_mappings = pickle.load(f)

decode_cols = {
    'Target_Program': 'Target_Program',
    'Target_Program_Discipline': 'Target_Program_Discipline',
    'Target_University': 'Target_University',
    'Target_Country': 'Target_Country'
}

def decode_freq_value(encoded_val, freq_dict):
    """
    Decode a frequency-encoded value by nearest match.
    """
    if pd.isna(encoded_val):
        return np.nan
    
    return min(
        freq_dict.items(),
        key=lambda x: abs(x[1] - encoded_val)
    )[0]

for df_col, mapping_key in decode_cols.items():
    if df_col in df.columns and mapping_key in freq_mappings:
        df[df_col.replace('_freq','')] = df[df_col].apply(
            lambda x: decode_freq_value(x, freq_mappings[mapping_key])
        )

df.head()


,Decision,Applicant_GPA,Decision_Year,Target_University_FSR_Score,Target_University_CPF_Score,Target_University_ISR_Score,Target_University_QS_Rank,Target_Degree_Type,Applicant_Status,Target_University_Status,Target_Program,Target_Program_Discipline,Target_University,Target_Country,Program_Admission_Rate,Program_x_QS_Rank,Applicant_ID,probability_of_acceptance
0,1,3.00,2025.0,53.0,100.0,23.5,49.0,1,1.0,1.0,Public Policy & Management,Politics & International Studies,"University of Maryland, Baltimore",China (Mainland),0.629944,-0.018297,1,0.837152
1,1,3.33,2025.0,17.6,79.1,95.5,65.0,1,1.0,1.0,Geography,Geography,lausanne,China (Mainland),0.570370,-0.006473,2,0.686783
2,1,3.85,2025.0,9.5,61.5,53.2,287.0,1,2.0,1.0,Computer Science,Computer Science & Information Systems,University of Arizona,United States of America,0.541090,0.004199,3,0.854386
3,1,3.70,2025.0,100.0,88.6,93.1,6.0,1,2.0,1.0,Food and Resource Economics,Economics & Econometrics,pratt,United Kingdom,0.670720,-0.065983,4,0.758738
4,0,3.94,2025.0,97.3,78.1,92.8,13.0,1,2.0,2.0,Film Studies,Performing Arts,University of Chicago,United States of America,0.597561,-0.004927,5,0.000000


University recommondation 

In [46]:
import pandas as pd

PROB_COL = "probability_of_acceptance"
TOP_N = 3
GPA_THRESHOLD = 0.1
FIXED_COLS = ["Target_Degree_Type"]

df.columns = df.columns.str.strip()

if PROB_COL not in df.columns:
    raise KeyError(
        f"Column '{PROB_COL}' not found in DataFrame. "
        f"Available columns: {df.columns.tolist()}"
    )

df_rejected = df[df["Decision"] == 0].copy()
recommendations = []

for applicant_id, df_app in df_rejected.groupby("Applicant_ID"):
    base = df_app.iloc[0]

    base_prob = base[PROB_COL]
    base_gpa = base["Applicant_GPA"]
    base_program = base["Target_Program"]
    base_uni = base["Target_University"]

    mask = df["Applicant_GPA"].between(
        base_gpa - GPA_THRESHOLD,
        base_gpa + GPA_THRESHOLD
    )

    for col in FIXED_COLS:
        mask &= df[col] == base[col]

    mask &= df["Target_University"] != base_uni

    mask &= df["Target_Program"] == base_program

    candidates = df[mask].copy()

    candidates = candidates[candidates[PROB_COL] > base_prob]

    if candidates.empty:
        continue


    candidates = candidates.sort_values(PROB_COL, ascending=False)

    candidates = candidates.drop_duplicates(
        subset="Target_University",
        keep="first"
    )

    topN = candidates.head(TOP_N)

    recommendations.append({
        "Applicant_ID": applicant_id,
        "Original_Target_University": base_uni,
        "Original_Target_Program": base_program,
        "Original_Acceptance_Probability": base_prob,
        "Recommended_Universities": list(topN["Target_University"]),
        "Recommended_Probabilities": list(topN[PROB_COL]),
    })

recommendations_df = pd.DataFrame(recommendations)

recommendations_df.head()


,Applicant_ID,Original_Target_University,Original_Target_Program,Original_Acceptance_Probability,Recommended_Universities,Recommended_Probabilities
0,5,University of Chicago,Film Studies,0.000000,"[University of Manchester, oxford, Brandeis Un...","[0.8684013724327088, 0.8606532454490662, 0.857..."
1,7,University of Chicago,Systems Engineering,0.690061,"[Vanderbilt University, University of Southern...","[0.8589922547340393, 0.8587213516235351, 0.849..."
2,9,"University of Maryland, Baltimore",Communication,0.823580,"[wake forest, ecole polytechnique federale de ...","[0.8704862713813781, 0.8666060924530029, 0.864..."
3,11,"University of California, Santa Barbara",Physics,0.833915,"[University of Virginia, Yale University, Univ...","[1.0, 1.0, 1.0]"
4,14,"University of Maryland, College Park",Physics,0.712223,"[University of Pennsylvania, University of Min...","[1.0, 1.0, 1.0]"


In [47]:

empty_count = recommendations_df["Recommended_Universities"].apply(lambda x: len(x) == 0).sum()
total_count = len(recommendations_df)
empty_percentage = (empty_count / total_count) * 100
print(f"Percentage of empty Recommended_Universities: {empty_percentage:.2f}%")

Percentage of empty Recommended_Universities: 0.00%


In [48]:
import pandas as pd

df_recs = recommendations_df.copy()

df_recs['Max_Recommended_Probability'] = df_recs['Recommended_Probabilities'].apply(lambda x: max(x))
df_recs['Improvement'] = df_recs['Max_Recommended_Probability'] - df_recs['Original_Acceptance_Probability']

average_improvement = df_recs['Improvement'].mean()

pct_better_option = (df_recs['Improvement'] > 0).mean() * 100
print(f"Percentage of applicants with a better option: {pct_better_option:.2f}%")


Percentage of applicants with a better option: 100.00%


defining a gpa threshhold based on our domain knowladge about universities 

In [49]:
import pandas as pd
import numpy as np


PROB_COL = "probability_of_acceptance"
TOP_N = 3
FIXED_COLS = ["Target_Degree_Type"]

def get_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df.columns = df.columns.str.strip()

required_cols = [
    PROB_COL,
    "Applicant_GPA",
    "Target_Program",
    "Target_University",
    "Target_University_QS_Rank",
    "Decision",
    "Applicant_ID"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

df["Target_University_QS_Rank"] = (
    df["Target_University_QS_Rank"]
    .astype(float)
    .round()
)

df_rejected = df[df["Decision"] == 0].copy()
recommendations = []

for applicant_id, df_app in df_rejected.groupby("Applicant_ID"):
    base = df_app.iloc[0]

    base_prob = base[PROB_COL]
    base_gpa = base["Applicant_GPA"]
    base_program = base["Target_Program"]
    base_uni = base["Target_University"]
    base_qs_rank = base["Target_University_QS_Rank"]

    gpa_threshold = get_gpa_threshold(base_qs_rank)

    mask = df["Applicant_GPA"].between(
        base_gpa - gpa_threshold,
        base_gpa + gpa_threshold
    )

    for col in FIXED_COLS:
        mask &= df[col] == base[col]

    mask &= df["Target_University"] != base_uni

    mask &= df["Target_Program"] == base_program

    candidates = df[mask].copy()

    candidates = candidates[candidates[PROB_COL] > base_prob]

    if candidates.empty:
        continue

    candidates = candidates.sort_values(PROB_COL, ascending=False)

    candidates = candidates.drop_duplicates(
        subset="Target_University",
        keep="first"
    )

    topN = candidates.head(TOP_N)

    recommendations.append({
        "Applicant_ID": applicant_id,
        "Original_Target_University": base_uni,
        "Original_Target_Program": base_program,
        "Original_Acceptance_Probability": base_prob,
        "QS_Rank_Based_GPA_Threshold": gpa_threshold,
        "Recommended_Universities": list(topN["Target_University"]),
        "Recommended_Probabilities": list(topN[PROB_COL]),
    })


recommendations_df = pd.DataFrame(recommendations)

if not recommendations_df.empty:
    recommendations_df["Max_Recommended_Probability"] = (
        recommendations_df["Recommended_Probabilities"].apply(max)
    )

    recommendations_df["Improvement"] = (
        recommendations_df["Max_Recommended_Probability"]
        - recommendations_df["Original_Acceptance_Probability"]
    )

    avg_improvement = recommendations_df["Improvement"].mean()
    pct_better_option = (recommendations_df["Improvement"] > 0).mean() * 100

    print(f"Average increase in acceptance probability: {avg_improvement:.3f}")
    print(f"Percentage of applicants with at least one better option: {pct_better_option:.2f}%")
else:
    print("No recommendations found.")


Average increase in acceptance probability: 0.687
Percentage of applicants with at least one better option: 100.00%


program recommandation

In [50]:
import pandas as pd
import numpy as np


PROB_COL = "probability_of_acceptance"
TOP_N = 3


def get_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25


program_recommendations = []

for applicant_id, df_app in df.groupby("Applicant_ID"):
    base = df_app.iloc[0]
    
    base_prob = base[PROB_COL]
    base_gpa = base["Applicant_GPA"]
    base_uni = base["Target_University"]
    base_discipline = base["Target_Program_Discipline"]
    base_program = base["Target_Program"]
    base_qs_rank = int(round(base["Target_University_QS_Rank"])) 
    

    gpa_threshold = get_gpa_threshold(base_qs_rank)
   
    mask = (
        (df["Target_University"] == base_uni) &
        (df["Target_Program_Discipline"] == base_discipline) &
        (df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold))
    )
    
    candidates = df[mask].copy()
 
    candidates = candidates[candidates["Target_Program"] != base_program]
 
    better_candidates = candidates[candidates[PROB_COL] > base_prob]

    better_candidates = better_candidates.sort_values(PROB_COL, ascending=False)\
                                         .drop_duplicates(subset="Target_Program")

    if not better_candidates.empty:
       
        topN = better_candidates.head(TOP_N)
        recommended_programs = list(topN["Target_Program"])
        recommended_probs = list(topN[PROB_COL])
    else:
        recommended_programs = [base_program]
        recommended_probs = [base_prob]

    program_recommendations.append({
        "Applicant_ID": applicant_id,
        "University": base_uni,
        "Target_Program_Discipline": base_discipline,
        "Original_Program": base_program,
        "Original_Probability": base_prob,
        "QS_Rank_Based_GPA_Threshold": gpa_threshold,  
        "Recommended_Programs": recommended_programs,
        "Recommended_Probabilities": recommended_probs
    })

program_recommendations_df = pd.DataFrame(program_recommendations)

program_recommendations_df.head()


,Applicant_ID,University,Target_Program_Discipline,Original_Program,Original_Probability,QS_Rank_Based_GPA_Threshold,Recommended_Programs,Recommended_Probabilities
0,1,"University of Maryland, Baltimore",Politics & International Studies,Public Policy & Management,0.837152,0.10,[Public Policy & Management],[0.8371523261070252]
1,2,lausanne,Geography,Geography,0.686783,0.15,[Geography],[0.6867833733558655]
2,3,University of Arizona,Computer Science & Information Systems,Computer Science,0.854386,0.20,[Computer Science],[0.8543859481811523]
3,4,pratt,Economics & Econometrics,Food and Resource Economics,0.758738,0.10,[Food and Resource Economics],[0.7587379932403564]
4,5,University of Chicago,Performing Arts,Film Studies,0.000000,0.10,[Film Studies],[0.0]


In [51]:
import pandas as pd

df_recs = program_recommendations_df.copy()

df_recs["Max_Recommended_Probability"] = (
    df_recs["Recommended_Probabilities"]
    .apply(lambda x: max(x) if len(x) > 0 else None)
)

df_recs["Improvement"] = (
    df_recs["Max_Recommended_Probability"]
    - df_recs["Original_Probability"]
)

average_improvement = df_recs["Improvement"].mean()
print(f"Average increase in acceptance probability: {average_improvement:.3f}")

pct_better_option = (df_recs["Improvement"] > 0).mean() * 100
print(f"Percentage of applicants with a better option: {pct_better_option:.2f}%")


Average increase in acceptance probability: 0.142
Percentage of applicants with a better option: 36.57%


In [52]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
TOP_N_VALUES = [1, 2, 3, 4]  

def get_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df_rejected = df[df["Decision"] == 0].copy()

results = []

for N in TOP_N_VALUES:
    improvements = []

    for applicant_id, df_app in df_rejected.groupby("Applicant_ID"):
        base = df_app.iloc[0]
        
        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_discipline = base["Target_Program_Discipline"]
        base_program = base["Target_Program"]
        base_qs_rank = int(round(base["Target_University_QS_Rank"]))
    
        gpa_threshold = get_gpa_threshold(base_qs_rank)
      
        mask = (
            (df["Target_University"] == base_uni) &
            (df["Target_Program_Discipline"] == base_discipline) &
            (df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold)) &
            (df["Target_Program"] != base_program)  
        )
        candidates = df[mask].copy()
        candidates = candidates[candidates[PROB_COL] > base_prob] 
        candidates = candidates.sort_values(PROB_COL, ascending=False).drop_duplicates(subset="Target_Program")
        
        if not candidates.empty:
            topN = candidates.head(N)
            improvements.append(topN[PROB_COL].max() - base_prob)
    
    if improvements:
        improvements = np.array(improvements)
        results.append({
            "TOP_N": N,
            "Average_Improvement": improvements.mean()
        })

topn_results_df = pd.DataFrame(results)
print(topn_results_df)


   TOP_N  Average_Improvement
0      1              0.50475
1      2              0.50475
2      3              0.50475
3      4              0.50475


"We tested several values of TOP_N to determine the optimal number of recommended alternative programs per applicant. The evaluation showed that the average improvement in acceptance probability remained constant at 0.509 for TOP_N values from 1 to 4, and 100% of applicants had at least one better option in all cases. To remain more precise in suggesting alternative programs within the same discipline, we chose TOP_N = 1, ensuring that only the single most promising alternative is recommended."

combination recommandation

In [53]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
TOP_N = 3
FIXED_COLS = ["Target_Degree_Type"]

def get_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()

df_rejected = df[df["Decision"] == 0].copy()
final_recommendations = []

for row_id, base in df_rejected.iterrows():

    base_prob = base[PROB_COL]
    base_gpa = base["Applicant_GPA"]
    base_program = base["Target_Program"]
    base_uni = base["Target_University"]
    base_qs_rank = base["Target_University_QS_Rank"]
    base_discipline = base["Target_Program_Discipline"]

    gpa_threshold = get_gpa_threshold(base_qs_rank)

    uni_mask = df["Applicant_GPA"].between(
        base_gpa - gpa_threshold,
        base_gpa + gpa_threshold
    )

    for col in FIXED_COLS:
        uni_mask &= df[col] == base[col]

    uni_mask &= df["Target_Program_Discipline"] == base_discipline
    uni_mask &= df["Target_University"] != base_uni

    uni_candidates = df[uni_mask].copy()
    uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]

    if uni_candidates.empty:
        continue

    uni_candidates = (
        uni_candidates
        .sort_values(PROB_COL, ascending=False)
        .drop_duplicates(subset="Target_University", keep="first")
    )

    best_uni_row = uni_candidates.iloc[0]
    best_uni = best_uni_row["Target_University"]
    uni_only_prob = best_uni_row[PROB_COL]

    final_program = base_program
    final_prob = uni_only_prob

    prog_mask = df["Applicant_GPA"].between(
        base_gpa - gpa_threshold,
        base_gpa + gpa_threshold
    )

    for col in FIXED_COLS:
        prog_mask &= df[col] == base[col]

    prog_mask &= df["Target_University"] == best_uni
    prog_mask &= df["Target_Program_Discipline"] == base_discipline

    prog_candidates = df[prog_mask].copy()

    if not prog_candidates.empty:
        best_prog_idx = prog_candidates[PROB_COL].idxmax()
        best_prog_row = prog_candidates.loc[best_prog_idx]

        if best_prog_row[PROB_COL] > uni_only_prob:
            final_program = best_prog_row["Target_Program"]
            final_prob = best_prog_row[PROB_COL]

    improvement = final_prob - base_prob

    final_recommendations.append({
        "Row_ID": row_id,
        "Original_Target_University": base_uni,
        "Original_Target_Program": base_program,
        "Original_Acceptance_Probability": base_prob,
        "Recommended_University": best_uni,
        "Recommended_Program": final_program,
        "Final_Acceptance_Probability": final_prob,
        "Improvement": improvement
    })

final_df = pd.DataFrame(final_recommendations)

if not final_df.empty:
    avg_improvement = final_df["Improvement"].mean()
    pct_better = (final_df["Improvement"] > 0).mean() * 100

    print(final_df.head(10))
    print(f"\nAverage increase in acceptance probability: {avg_improvement:.3f}")
    print(f"Percentage of rejected rows with improvement: {pct_better:.2f}%")
else:
    print("No recommendations found.")


   Row_ID               Original_Target_University  \
0       4                    University of Chicago   
1       6                    University of Chicago   
2       8        University of Maryland, Baltimore   
3      10  University of California, Santa Barbara   
4      13     University of Maryland, College Park   
5      14                   University of Waterloo   
6      15                                  houston   
7      16           University of Colorado Boulder   
8      17        University of Southern California   
9      18                 University of Manchester   

         Original_Target_Program  Original_Acceptance_Probability  \
0                   Film Studies                         0.000000   
1            Systems Engineering                         0.690061   
2                  Communication                         0.823580   
3                        Physics                         0.833915   
4                        Physics                         0.7

In [54]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
TOP_N = 5
GPA_MULTIPLIERS = np.arange(1.0, 1.25, 0.02)  


def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25


df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()

df_rejected = df[df["Decision"] == 0].copy()

grid_results = []

for gpa_mult in GPA_MULTIPLIERS:

    final_recommendations = []

    for _, base in df_rejected.iterrows():

        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]

        gpa_threshold = base_gpa_threshold(base_qs_rank) * gpa_mult

        uni_mask = df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold)
        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]

        if uni_candidates.empty:
            continue
        uni_candidates = uni_candidates.sort_values(PROB_COL, ascending=False).head(10)
        uni_candidates = uni_candidates.drop_duplicates(subset="Target_University", keep="first").head(TOP_N)

        best_final_prob = uni_candidates.iloc[0][PROB_COL]
        best_final_program = None
        best_final_uni = uni_candidates.iloc[0]["Target_University"]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold)
            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_idx = prog_candidates[PROB_COL].idxmax()
            best_prog_row = prog_candidates.loc[best_prog_idx]

            if best_prog_row[PROB_COL] > best_final_prob:
                best_final_prob = best_prog_row[PROB_COL]
                best_final_program = best_prog_row["Target_Program"]
                best_final_uni = uni

        improvement = best_final_prob - base_prob
        final_recommendations.append(improvement)

    if final_recommendations:
        avg_improvement = np.mean(final_recommendations)
        pct_better = (np.array(final_recommendations) > 0).mean() * 100

        grid_results.append({
            "GPA_Multiplier": gpa_mult,
            "Average_Improvement": avg_improvement,
            "Pct_With_Improvement": pct_better
        })
        
grid_df = pd.DataFrame(grid_results).sort_values("Average_Improvement", ascending=False)
print(grid_df)


    GPA_Multiplier  Average_Improvement  Pct_With_Improvement
12            1.24             0.706564                 100.0
11            1.22             0.706563                 100.0
10            1.20             0.706499                 100.0
8             1.16             0.705334                 100.0
9             1.18             0.705334                 100.0
7             1.14             0.705190                 100.0
6             1.12             0.705034                 100.0
5             1.10             0.704758                 100.0
4             1.08             0.704478                 100.0
2             1.04             0.704158                 100.0
3             1.06             0.704158                 100.0
1             1.02             0.703913                 100.0
0             1.00             0.703913                 100.0


In [55]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
TOP_N = 5
GPA_MULTIPLIERS = np.arange(1.26, 1.51, 0.02) 

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()

df_rejected = df[df["Decision"] == 0].copy()

grid_results = []

for gpa_mult in GPA_MULTIPLIERS:

    final_recommendations = []

    for _, base in df_rejected.iterrows():

        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]

        gpa_threshold = base_gpa_threshold(base_qs_rank) * gpa_mult

        uni_mask = df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold)
        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]

        if uni_candidates.empty:
            continue

        uni_candidates = uni_candidates.sort_values(PROB_COL, ascending=False).head(10)
        uni_candidates = uni_candidates.drop_duplicates(subset="Target_University", keep="first").head(TOP_N)

        best_final_prob = uni_candidates.iloc[0][PROB_COL]
        best_final_program = None
        best_final_uni = uni_candidates.iloc[0]["Target_University"]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold)
            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_idx = prog_candidates[PROB_COL].idxmax()
            best_prog_row = prog_candidates.loc[best_prog_idx]

            if best_prog_row[PROB_COL] > best_final_prob:
                best_final_prob = best_prog_row[PROB_COL]
                best_final_program = best_prog_row["Target_Program"]
                best_final_uni = uni

        improvement = best_final_prob - base_prob
        final_recommendations.append(improvement)

    if final_recommendations:
        avg_improvement = np.mean(final_recommendations)
        pct_better = (np.array(final_recommendations) > 0).mean() * 100

        grid_results.append({
            "GPA_Multiplier": gpa_mult,
            "Average_Improvement": avg_improvement,
            "Pct_With_Improvement": pct_better
        })

grid_df = pd.DataFrame(grid_results).sort_values("Average_Improvement", ascending=False)
print(grid_df)


    GPA_Multiplier  Average_Improvement  Pct_With_Improvement
12            1.50             0.708775                 100.0
11            1.48             0.708550                 100.0
10            1.46             0.708392                 100.0
9             1.44             0.708383                 100.0
8             1.42             0.708263                 100.0
7             1.40             0.708259                 100.0
6             1.38             0.708024                 100.0
5             1.36             0.707987                 100.0
4             1.34             0.707904                 100.0
3             1.32             0.707468                 100.0
2             1.30             0.707059                 100.0
1             1.28             0.706953                 100.0
0             1.26             0.706573                 100.0


In [56]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
TOP_N_LIST = [5, 6, 7, 8, 9, 10]
GPA_MULT = 1

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.10 * GPA_MULT
    elif qs_rank <= 150:
        return 0.15 * GPA_MULT
    elif qs_rank <= 300:
        return 0.20 * GPA_MULT
    else:
        return 0.25 * GPA_MULT

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()
df_rejected = df[df["Decision"] == 0].copy()

def compute_improvement(TOP_N):
    improvements = []

    for _, base in df_rejected.iterrows():
        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]

        gpa_threshold = base_gpa_threshold(base_qs_rank)

        uni_mask = df["Applicant_GPA"].between(
            base_gpa - gpa_threshold,
            base_gpa + gpa_threshold
        )
        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]
        if uni_candidates.empty:
            continue

        uni_candidates = (
            uni_candidates
            .sort_values(PROB_COL, ascending=False)
            .drop_duplicates(subset="Target_University", keep="first")
            .head(TOP_N)
        )

        best_final_prob = uni_candidates.iloc[0][PROB_COL]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(
                base_gpa - gpa_threshold,
                base_gpa + gpa_threshold
            )
            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_prob = prog_candidates[PROB_COL].max()
            if best_prog_prob > best_final_prob:
                best_final_prob = best_prog_prob

        improvements.append(best_final_prob - base_prob)

    if improvements:
        return np.mean(improvements), (np.array(improvements) > 0).mean() * 100
    else:
        return 0, 0

grid_results = []
for TOP_N in TOP_N_LIST:
    avg_imp, pct = compute_improvement(TOP_N)
    grid_results.append({
        "TOP_N": TOP_N,
        "GPA_Multiplier": GPA_MULT,
        "Average_Improvement": avg_imp,
        "Pct_With_Improvement": pct
    })

grid_df = pd.DataFrame(grid_results).sort_values("Average_Improvement", ascending=False)
print(grid_df)

best_row = grid_df.iloc[0]
print("\n=== Best TOP_N with GPA multiplier 1.5 ===")
print(best_row)


   TOP_N  GPA_Multiplier  Average_Improvement  Pct_With_Improvement
0      5               1             0.703913                 100.0
1      6               1             0.703913                 100.0
2      7               1             0.703913                 100.0
3      8               1             0.703913                 100.0
4      9               1             0.703913                 100.0
5     10               1             0.703913                 100.0

=== Best TOP_N with GPA multiplier 1.5 ===
TOP_N                     5.000000
GPA_Multiplier            1.000000
Average_Improvement       0.703913
Pct_With_Improvement    100.000000
Name: 0, dtype: float64


In [57]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
N = 5  

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.1
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()
df_rejected = df[df["Decision"] == 0].copy()

def compute_improvement(N):
    improvements = []

    for _, base in df_rejected.iterrows():
        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]

        gpa_threshold = base_gpa_threshold(base_qs_rank)

        uni_mask = df["Applicant_GPA"].between(
            base_gpa - gpa_threshold,
            base_gpa + gpa_threshold
        )
        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]
        if uni_candidates.empty:
            continue

        uni_candidates = (
            uni_candidates
            .sort_values(PROB_COL, ascending=False)
            .drop_duplicates(subset="Target_University", keep="first")
            .head(N)
        )

        best_final_prob = uni_candidates.iloc[0][PROB_COL]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(
                base_gpa - gpa_threshold,
                base_gpa + gpa_threshold
            )
            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_prob = prog_candidates[PROB_COL].max()
            if best_prog_prob > best_final_prob:
                best_final_prob = best_prog_prob

        improvements.append(best_final_prob - base_prob)

    if improvements:
        avg_improvement = np.mean(improvements)
        pct_better = (np.array(improvements) > 0).mean() * 100
    else:
        avg_improvement = 0
        pct_better = 0

    return avg_improvement, pct_better

avg_imp, pct = compute_improvement(N)
print(f"Average Improvement: {avg_imp:.3f}")
print(f"Percentage with Improvement: {pct:.2f}%")


Average Improvement: 0.704
Percentage with Improvement: 100.00%


In [58]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
N = 5  

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.1
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()
df_rejected = df[df["Decision"] == 0].copy()

def compute_improvement(N):
    improvements = []

    for _, base in df_rejected.iterrows():
        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]
        base_country = base["Target_Country"]

        gpa_threshold = base_gpa_threshold(base_qs_rank)

        uni_mask = df["Applicant_GPA"].between(
            base_gpa - gpa_threshold,
            base_gpa + gpa_threshold
        )
        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]
        if uni_candidates.empty:
            continue
        
        uni_candidates["country_priority"] = (
            uni_candidates["Target_Country"] == base_country
        ).astype(int)

        uni_candidates = (
            uni_candidates
            .sort_values(["country_priority", PROB_COL], ascending=[False, False])
            .drop_duplicates(subset="Target_University", keep="first")
            .head(N)
        )

        best_final_prob = uni_candidates.iloc[0][PROB_COL]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(
                base_gpa - gpa_threshold,
                base_gpa + gpa_threshold
            )
            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_prob = prog_candidates[PROB_COL].max()
            if best_prog_prob > best_final_prob:
                best_final_prob = best_prog_prob

        improvements.append(best_final_prob - base_prob)

    if improvements:
        avg_improvement = np.mean(improvements)
        pct_better = (np.array(improvements) > 0).mean() * 100
    else:
        avg_improvement = 0
        pct_better = 0

    return avg_improvement, pct_better

avg_imp, pct = compute_improvement(N)
print(f"=== Results with N = {N} ===")
print(f"Average Improvement: {avg_imp:.3f}")
print(f"Percentage with Improvement: {pct:.2f}%")


=== Results with N = 5 ===
Average Improvement: 0.701
Percentage with Improvement: 100.00%


For the `Target_Country`, I applied a **hard filter and a priority flag** to ensure relevance while slightly favoring universities in the same country. First, all candidate universities must be **different from the applicant’s original university**, ensuring that only true alternatives are considered. Then, a `country_priority` column is created where universities in the **same country as the original university** receive a value of `1` and others `0`. This flag is used as a **tertiary sorting criterion**, after probability of acceptance and GPA proximity, so that among equally strong candidates, universities in the same country are preferred. This approach maintains strict relevance while subtly increasing the likelihood of recommending geographically closer or contextually more relevant universities.


In [59]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
N = 5  

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.1
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()
df_rejected = df[df["Decision"] == 0].copy()

def compute_improvement(N):
    improvements = []

    for _, base in df_rejected.iterrows():
        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]
        base_country = base["Target_Country"]

        gpa_threshold = base_gpa_threshold(base_qs_rank)

        uni_mask = df["Applicant_GPA"].between(
            base_gpa - gpa_threshold,
            base_gpa + gpa_threshold
        )

        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]
        if uni_candidates.empty:
            continue

        uni_candidates["gpa_score"] = (
            1 - abs(uni_candidates["Applicant_GPA"] - base_gpa) / gpa_threshold
        ).clip(0, 1)

        uni_candidates["country_priority"] = (
            uni_candidates["Target_Country"] == base_country
        ).astype(int)

        uni_candidates = (
            uni_candidates
            .sort_values([PROB_COL, "gpa_score", "country_priority"],
                         ascending=[False, False, False])
            .drop_duplicates(subset="Target_University", keep="first")
            .head(N)
        )

        best_final_prob = uni_candidates.iloc[0][PROB_COL]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(
                base_gpa - gpa_threshold,
                base_gpa + gpa_threshold
            )

            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_prob = prog_candidates[PROB_COL].max()
            if best_prog_prob > best_final_prob:
                best_final_prob = best_prog_prob

        improvements.append(best_final_prob - base_prob)

    if improvements:
        avg_improvement = np.mean(improvements)
        pct_better = (np.array(improvements) > 0).mean() * 100
    else:
        avg_improvement = 0
        pct_better = 0

    return avg_improvement, pct_better

avg_imp, pct = compute_improvement(N)
print(f"=== Results with N = {N} ===")
print(f"Average Improvement: {avg_imp:.3f}")
print(f"Percentage with Improvement: {pct:.2f}%")


=== Results with N = 5 ===
Average Improvement: 0.704
Percentage with Improvement: 100.00%


In the updated code, I introduced a **GPA proximity score** to prioritize candidates whose GPA is closer to the applicant’s own GPA, while still respecting the original hard threshold. Specifically, for each candidate within the threshold, the score is computed as `1 - abs(candidate_GPA - base_GPA)/gpa_threshold`, ensuring that candidates at the edge of the threshold receive a lower score than those near the applicant’s GPA. This score is then used as a **secondary sorting criterion** after the probability of acceptance, so that among candidates with similar probabilities, those with closer GPAs are preferred. This approach improves the relevance and personalization of recommendations by subtly favoring universities and programs where the applicant’s academic profile more closely matches the typical accepted profile, without including candidates outside the allowable GPA range.


Final combined university-program code 

In [60]:
import pandas as pd
import numpy as np

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
N = 5 

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.1
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

status_match_dict = {
    1: [1, 2],   
    2: [1, 2],   
    3: [1, 2, 3] 
}

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_Program_Discipline"] = df["Target_Program_Discipline"].str.strip().str.lower()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()
df_rejected = df[df["Decision"] == 0].copy()

def compute_improvement(N):
    improvements = []

    for _, base in df_rejected.iterrows():
        base_prob = base[PROB_COL]
        base_gpa = base["Applicant_GPA"]
        base_uni = base["Target_University"]
        base_qs_rank = base["Target_University_QS_Rank"]
        base_discipline = base["Target_Program_Discipline"]
        base_country = base["Target_Country"]
        base_status = base.get("Target_University_Status", None)

        gpa_threshold = base_gpa_threshold(base_qs_rank)

        uni_mask = df["Applicant_GPA"].between(
            base_gpa - gpa_threshold,
            base_gpa + gpa_threshold
        )

        for col in FIXED_COLS:
            uni_mask &= df[col] == base[col]

        uni_mask &= df["Target_Program_Discipline"] == base_discipline
        uni_mask &= df["Target_University"] != base_uni

        if base_status is not None and "Target_University_Status" in df.columns:
            allowed_statuses = status_match_dict.get(base_status, [base_status])
            uni_mask &= df["Target_University_Status"].isin(allowed_statuses)

        uni_candidates = df[uni_mask].copy()
        uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]
        if uni_candidates.empty:
            continue

        uni_candidates["gpa_score"] = (
            1 - abs(uni_candidates["Applicant_GPA"] - base_gpa) / gpa_threshold
        ).clip(0, 1)

        uni_candidates["country_priority"] = (
            uni_candidates["Target_Country"] == base_country
        ).astype(int)

        uni_candidates = (
            uni_candidates
            .sort_values([PROB_COL, "gpa_score", "country_priority"],
                         ascending=[False, False, False])
            .drop_duplicates(subset="Target_University", keep="first")
            .head(N)
        )

        best_final_prob = uni_candidates.iloc[0][PROB_COL]

        for _, uni_row in uni_candidates.iterrows():
            uni = uni_row["Target_University"]

            prog_mask = df["Applicant_GPA"].between(
                base_gpa - gpa_threshold,
                base_gpa + gpa_threshold
            )

            for col in FIXED_COLS:
                prog_mask &= df[col] == base[col]

            prog_mask &= df["Target_University"] == uni
            prog_mask &= df["Target_Program_Discipline"] == base_discipline

            prog_candidates = df[prog_mask].copy()
            if prog_candidates.empty:
                continue

            best_prog_prob = prog_candidates[PROB_COL].max()
            best_final_prob = max(best_final_prob, best_prog_prob)

        improvements.append(best_final_prob - base_prob)

    if improvements:
        avg_improvement = np.mean(improvements)
        pct_better = (np.array(improvements) > 0).mean() * 100
    else:
        avg_improvement = 0
        pct_better = 0

    return avg_improvement, pct_better

avg_imp, pct = compute_improvement(N)
print(f"=== Results with N = {N} ===")
print(f"Average Improvement: {avg_imp:.3f}")
print(f"Percentage with Improvement: {pct:.2f}%")


=== Results with N = 5 ===
Average Improvement: 0.704
Percentage with Improvement: 100.00%




To ensure recommendations are financially realistic, we incorporated a filter based on `Target_University_Status`, which reflects the tuition and cost structure of universities:

Public can only be recommended Public or Private NFP universities. Private FP schools are excluded because these applicants likely cannot afford the highest tuition.  
Private NFP can be recommended Public or Private NFP universities, but not Private FP, reflecting medium affordability.  
Private FP can be recommended any type (Public, Private NFP, Private FP), reflecting the highest affordability.

In [61]:

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]

status_type_mapping = {
    'Public': 1,
    'Private not for Profit': 2,
    'Private for Profit': 3
}

status_match_dict = {
   1: [1, 2],   
    2: [1, 2],  
    3: [1, 2, 3] 
}

def base_gpa_threshold(qs_rank):
    if qs_rank <= 50:
        return 0.15
    elif qs_rank <= 150:
        return 0.20
    else:
        return 0.25


df_rejected = df[df["Decision"] == 0].copy()

excluded_applicants = []  

for _, base in df_rejected.iterrows():
    base_gpa = base["Applicant_GPA"]
    base_uni = base["Target_University"]
    base_qs_rank = base["Target_University_QS_Rank"]
    base_discipline = base["Target_Program_Discipline"]
    base_status = status_type_mapping.get(base["Target_University_Status"], None)

    gpa_threshold = base_gpa_threshold(base_qs_rank)

    uni_mask = df["Applicant_GPA"].between(
        base_gpa - gpa_threshold,
        base_gpa + gpa_threshold
    )

    for col in FIXED_COLS:
        uni_mask &= df[col] == base[col]

    uni_mask &= df["Target_Program_Discipline"] == base_discipline
    uni_mask &= df["Target_University"] != base_uni

    if base_status is not None:
        allowed_statuses = status_match_dict[base_status]
        uni_mask &= df["Target_University_Status"].map(status_type_mapping).isin(allowed_statuses)

    uni_candidates = df[uni_mask].copy()
    uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base[PROB_COL]]

    if uni_candidates.empty:
        excluded_applicants.append(base["Applicant_ID"])


num_excluded = len(excluded_applicants)
num_total_rejected = df_rejected.shape[0]
num_considered = num_total_rejected - num_excluded

print(f"Total rejected applicants: {num_total_rejected}")
print(f"Rejected applicants considered for improvement: {num_considered}")
print(f"Rejected applicants excluded (no candidates found): {num_excluded}")


Total rejected applicants: 6028
Rejected applicants considered for improvement: 5590
Rejected applicants excluded (no candidates found): 438


In [62]:
import numpy as np

improvements = np.array(improvements)

num_total_rejected = df[df["Decision"] == 0].shape[0]

num_improved = (improvements > 0).sum()

pct_improved_all = (num_improved / num_total_rejected) * 100

avg_improvement_all = improvements.sum() / num_total_rejected

print(f"Percentage of all rejected applicants with improvement: {pct_improved_all:.2f}%")
print(f"Average improvement across all rejected applicants: {avg_improvement_all:.3f}")


Percentage of all rejected applicants with improvement: 44.26%
Average improvement across all rejected applicants: 0.223


final improved university recommondation only 

In [63]:
import pandas as pd
import numpy as np


PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
TOP_N = 5 

def gpa_threshold_by_qs(qs_rank):
    if qs_rank <= 50:
        return 0.1
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

status_match_dict = {
    1: [1, 2],  
    2: [1, 2],   
    3: [1, 2, 3] 
}

df = df.copy()
df.columns = df.columns.str.strip()
df["Target_University_QS_Rank"] = df["Target_University_QS_Rank"].astype(float).round()
df_rejected = df[df["Decision"] == 0].copy()

recommendations = []

for _, base in df_rejected.iterrows():
    base_prob = base[PROB_COL]
    base_gpa = base["Applicant_GPA"]
    base_uni = base["Target_University"]
    base_country = base["Target_Country"]
    base_program = base["Target_Program"]  
    base_qs_rank = base["Target_University_QS_Rank"]
    base_status = base.get("Target_University_Status", None)

    gpa_thresh = gpa_threshold_by_qs(base_qs_rank)

    uni_mask = df["Applicant_GPA"].between(
        base_gpa - gpa_thresh,
        base_gpa + gpa_thresh
    )

    for col in FIXED_COLS:
        uni_mask &= df[col] == base[col]

    uni_mask &= df["Target_Program"] == base_program

    uni_mask &= df["Target_University"] != base_uni

    if base_status is not None and "Target_University_Status" in df.columns:
        allowed_statuses = status_match_dict.get(base_status, [base_status])
        uni_mask &= df["Target_University_Status"].isin(allowed_statuses)

    uni_candidates = df[uni_mask].copy()
    uni_candidates = uni_candidates[uni_candidates[PROB_COL] > base_prob]
    if uni_candidates.empty:
        continue

    uni_candidates["gpa_score"] = (
        1 - abs(uni_candidates["Applicant_GPA"] - base_gpa) / gpa_thresh
    ).clip(0, 1)
    uni_candidates["country_priority"] = (uni_candidates["Target_Country"] == base_country).astype(int)

    uni_candidates = (
        uni_candidates
        .sort_values([PROB_COL, "gpa_score", "country_priority"], ascending=[False, False, False])
        .drop_duplicates(subset="Target_University", keep="first")
        .head(TOP_N)
    )

    recommendations.append({
        "Applicant_ID": base["Applicant_ID"],
        "Original_Target_University": base_uni,
        "Original_Target_Program": base_program,
        "Original_Probability": base_prob,
        "QS_GPA_Threshold": gpa_thresh,
        "Recommended_Universities": list(uni_candidates["Target_University"]),
        "Recommended_Probabilities": list(uni_candidates[PROB_COL]),
    })

recommendations_df = pd.DataFrame(recommendations)

if not recommendations_df.empty:
    recommendations_df["Max_Recommended_Probability"] = recommendations_df["Recommended_Probabilities"].apply(max)
    recommendations_df["Improvement"] = (
        recommendations_df["Max_Recommended_Probability"] - recommendations_df["Original_Probability"]
    )

    avg_improvement = recommendations_df["Improvement"].mean()
    pct_better_option = (recommendations_df["Improvement"] > 0).mean() * 100

    print(f"Average increase in acceptance probability: {avg_improvement:.3f}")
    print(f"Percentage of applicants with at least one better option: {pct_better_option:.2f}%")
else:
    print("No recommendations found.")


Average increase in acceptance probability: 0.687
Percentage of applicants with at least one better option: 100.00%


final improved program recommnadation only based on the findings on the combination recommonder

In [64]:

PROB_COL = "probability_of_acceptance"
FIXED_COLS = ["Target_Degree_Type"]
TOP_N = 5  


def gpa_threshold_by_qs(qs_rank):
    if qs_rank <= 50:
        return 0.1
    elif qs_rank <= 150:
        return 0.15
    elif qs_rank <= 300:
        return 0.20
    else:
        return 0.25

program_recommendations = []

for idx, base in df.iterrows():
    base_prob = base[PROB_COL]
    base_gpa = base["Applicant_GPA"]
    base_uni = base["Target_University"]
    base_discipline = base["Target_Program_Discipline"]
    base_program = base["Target_Program"]
    base_country = base["Target_Country"]
    base_qs_rank = int(round(base["Target_University_QS_Rank"]))

    gpa_threshold = gpa_threshold_by_qs(base_qs_rank)
    mask = (
        (df["Target_University"] == base_uni) &
        (df["Target_Program_Discipline"] == base_discipline) &
        (df["Applicant_GPA"].between(base_gpa - gpa_threshold, base_gpa + gpa_threshold))
    )

    candidates = df[mask].copy()

    candidates = candidates[candidates["Target_Program"] != base_program]

    candidates = candidates[candidates[PROB_COL] > base_prob]

    if candidates.empty:
        
        recommended_programs = [base_program]
        recommended_probs = [base_prob]
    else:
        candidates['gpa_proximity'] = 1 - abs(candidates['Applicant_GPA'] - base_gpa) / gpa_threshold
        candidates['gpa_proximity'] = candidates['gpa_proximity'].clip(0, 1)

        candidates = candidates.sort_values([PROB_COL, 'gpa_proximity'], ascending=[False, False])

        topN = candidates.drop_duplicates(subset="Target_Program").head(TOP_N)

        recommended_programs = list(topN["Target_Program"])
        recommended_probs = list(topN[PROB_COL])

    program_recommendations.append({
        "Row_Index": idx,
        "University": base_uni,
        "Target_Program_Discipline": base_discipline,
        "Original_Program": base_program,
        "Original_Probability": base_prob,
        "QS_Rank_Based_GPA_Threshold": gpa_threshold,
        "Recommended_Programs": recommended_programs,
        "Recommended_Probabilities": recommended_probs
    })

program_recommendations_df = pd.DataFrame(program_recommendations)

program_recommendations_df["Max_Recommended_Probability"] = program_recommendations_df["Recommended_Probabilities"].apply(lambda x: max(x) if len(x) > 0 else None)
program_recommendations_df["Improvement"] = program_recommendations_df["Max_Recommended_Probability"] - program_recommendations_df["Original_Probability"]

average_improvement = program_recommendations_df["Improvement"].mean()

pct_better_option = (program_recommendations_df["Improvement"] > 0).mean() * 100

print(f"Average increase in acceptance probability: {average_improvement:.3f}")
print(f"Percentage of rows with a better option: {pct_better_option:.2f}%")

program_recommendations_df.head()


Average increase in acceptance probability: 0.142
Percentage of rows with a better option: 36.57%


,Row_Index,University,Target_Program_Discipline,Original_Program,Original_Probability,QS_Rank_Based_GPA_Threshold,Recommended_Programs,Recommended_Probabilities,Max_Recommended_Probability,Improvement
0,0,"University of Maryland, Baltimore",politics & international studies,Public Policy & Management,0.837152,0.10,[Public Policy & Management],[0.8371523261070252],0.837152,0.0
1,1,lausanne,geography,Geography,0.686783,0.15,[Geography],[0.6867833733558655],0.686783,0.0
2,2,University of Arizona,computer science & information systems,Computer Science,0.854386,0.20,[Computer Science],[0.8543859481811523],0.854386,0.0
3,3,pratt,economics & econometrics,Food and Resource Economics,0.758738,0.10,[Food and Resource Economics],[0.7587379932403564],0.758738,0.0
4,4,University of Chicago,performing arts,Film Studies,0.000000,0.10,[Film Studies],[0.0],0.000000,0.0


In [65]:

max_improvement = program_recommendations_df["Improvement"].max()

max_improvement_rows = program_recommendations_df[program_recommendations_df["Improvement"] == max_improvement]

print(f"Maximum improvement in acceptance probability: {max_improvement:.3f}")
print("Row(s) with maximum improvement:")
print(max_improvement_rows[[
    "Row_Index", 
    "University", 
    "Original_Program", 
    "Original_Probability", 
    "Recommended_Programs", 
    "Recommended_Probabilities", 
    "Improvement"
]])


Maximum improvement in acceptance probability: 1.000
Row(s) with maximum improvement:
       Row_Index                  University             Original_Program  \
138          138    University of Washington                     Robotics   
295          295             Yale University                    Economics   
515          515    University of Manchester                    Economics   
718          718  Carnegie Mellon University   Human-Computer Interaction   
890          890     Northeastern University                     Robotics   
...          ...                         ...                          ...   
12497      12497                      oxford  Food and Resource Economics   
12584      12584                       pratt  Food and Resource Economics   
12780      12780       University of Alberta  Food and Resource Economics   
12857      12857      University of Michigan          Physics & Astronomy   
12933      12933          University of Utah  Food and Resource Eco